# Data Exploration: Three NL Aces in 2026

**Objective:** Understand the scope, structure, and quality of Statcast pitch-by-pitch data for three elite National League pitchers: Shohei Ohtani (LAD), Cristopher Sanchez (PHI), and Jacob Misiorowski (MIL).

**Questions this notebook answers:**
- How much data do we have for each pitcher?
- What pitch types does each throw, and how often?
- What is the basic velocity, spin, and movement profile?
- Are there any data quality issues (missing values, outliers)?

In [ ]:
# ---- Imports ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from analysis_utils import load_all, set_style

set_style()
dfs = load_all()

for name, df in dfs.items():
    print(f"  {name:12s}  {len(df):5d} pitches  {df['game_date'].nunique():2d} games  "
          f"{df['game_date'].min()} to {df['game_date'].max()}")

In [ ]:
# ---- Basic Statistics per Pitcher ----
for name, df in dfs.items():
    vel = df['release_speed'].dropna()
    spin = df['release_spin_rate'].dropna()
    in_zone = df['zone'].dropna()
    zone_pct = (in_zone <= 9).sum() / len(in_zone) * 100
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Pitches: {len(df)}  |  Games: {df['game_date'].nunique()}")
    print(f"  Velocity: {vel.mean():.1f} +/- {vel.std():.1f} mph  (max {vel.max():.1f})")
    print(f"  Spin Rate: {spin.mean():.0f} +/- {spin.std():.0f} rpm")
    print(f"  Zone%: {zone_pct:.1f}%")
    print(f"  Pitch Types: {sorted(df['pitch_type'].dropna().unique())}")
    print(f"  Top 3 Pitch Usage:")
    for pt, ct in df['pitch_type'].value_counts().head(3).items():
        print(f"    {pt}: {ct} ({ct/len(df)*100:.1f}%)")

In [ ]:
# ---- Missing Data Check ----
cols = ['release_speed','release_spin_rate','pfx_x','pfx_z',
        'release_extension','zone','estimated_woba_using_speedangle',
        'delta_pitcher_run_exp','launch_speed','launch_angle']
for name, df in dfs.items():
    miss = df[cols].isna().sum()
    miss = miss[miss > 0]
    if len(miss) > 0:
        print(f"\n{name}:")
        for c, v in miss.items():
            print(f"  {c:40s} {v:5d} missing ({v/len(df)*100:.1f}%)")
    else:
        print(f"{name}: No missing values in key columns")

In [ ]:
# ---- Pitch Arsenal Pie Charts ----
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    counts = df['pitch_type'].value_counts()
    colors = plt.cm.Set2(np.linspace(0, 1, len(counts)))
    wedges, texts, autotexts = ax.pie(
        counts.values, labels=counts.index, autopct='%1.1f%%',
        colors=colors, startangle=90, textprops={'fontsize': 10})
    ax.set_title(f"{name}", fontweight='bold', fontsize=13)
plt.suptitle("Pitch Arsenal Composition", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('figures/arsenal_pie.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Velocity Distribution ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, (name, df) in zip(axes, dfs.items()):
    pts = sorted(df['pitch_type'].dropna().unique())
    data = [df[df['pitch_type']==pt]['release_speed'].dropna().values for pt in pts]
    bp = ax.boxplot(data, label=pts, patch_artist=True)
    for patch, color in zip(bp['boxes'], plt.cm.Set2(np.linspace(0,1,len(pts)))):
        patch.set_facecolor(color)
    ax.set_title(name, fontweight='bold')
    ax.set_ylabel("Velocity (mph)")
    ax.set_ylim(70, 108)
plt.suptitle("Velocity Distribution by Pitch Type", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/velocity_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Movement Profiles ----
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (name, df) in zip(axes, dfs.items()):
    for pt in df['pitch_type'].dropna().unique():
        sub = df[df['pitch_type']==pt]
        ax.scatter(sub['pfx_x'], sub['pfx_z'], alpha=0.4, s=15, label=pt)
    ax.axhline(0, color='gray', ls='--', lw=0.5)
    ax.axvline(0, color='gray', ls='--', lw=0.5)
    ax.set_xlabel("Horizontal Break (in)")
    ax.set_ylabel("Vertical Break (in)")
    ax.set_title(name, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlim(-25, 25); ax.set_ylim(-25, 25)
plt.suptitle("Pitch Movement Profiles", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/movement_profiles.png', dpi=150, bbox_inches='tight')
plt.show()